# 3 · Guainía: forest height from the tomograms

**Pipeline:** 0 · Data access → 1 · Stack → 2 · Tomography → **3 · Forest height** → 4 · Polarimetric analysis

Input: tomograms and coherence matrices from notebook 2, through the manifest. Heights are relative to the SKP-calibrated ground ($z=0$).

| estimator | ground | top | height |
|---|---|---|---|
| energy envelope | $z=0$ | height below which 95 % of the profile energy lies | $H_{95}$ |
| percentile envelope | 5 % energy height | 95 % energy height | $H_{95} - H_5$ |
| min–max edge | profile peak | highest height above the peak where the min–max-scaled profile ≥ `EDGE_LEVEL` | $z_{edge} - z_{peak}$ |
| canopy top | $z=0$ | highest height contiguous with the peak within `TOP_DB` of the maximum | $h_{top}$ |
| phase centre | $z=0$ | – | $z_{pc}$: median over baselines of $-\arg\gamma/\Delta k_z$ (a centroid, below the top) |
| RVoG | $z=0$ | volume height of the Random-Volume-over-Ground model fitted to all pairs | $h_v$ |

With a vertical resolution of about 20 m (notebook 1), the Capon estimators are reliable for forests taller than roughly half the resolution. Below that, ground and canopy are not separated.

**Reference.** `Forest_Height_Guainia.tif` is a 30 m canopy-height map (EPSG:4326, uint8 metres) using the GLAD coding: 0 = no trees, 101 = water, 102 = snow/ice, 103 = no data. It is pooled onto the coherence grid over about the multilook footprint and used to assess every estimator. The best estimator also gets a bias correction.

**Steps:** load → reference on the radar grid → quality mask → heights → maps and distributions → example profiles → assessment against the reference → bias correction → geographic maps and GeoTIFF → optional biomass → save.

Maps in radar geometry are drawn at true ground scale. x is ground range and y is azimuth, with the first line at the top. For this descending pass that is roughly north-up and east-right.

## 1. Configuration
`REFERENCE_POOL` is the pooling size in coherence cells. The coherence grid is about 95 m, so 3 × 3 cells ≈ 280 m, slightly more than the 200 m multilook window plus geolocation error. The assessment is restricted to **forest** cells: no water in the footprint and less than `FOREST_MAX_NONFOREST` of its 30 m pixels without trees.

In [ ]:
import pathlib, sys
SITE_DIR = pathlib.Path.cwd()                       # the Guainia/ folder (notebooks run from here)
if not (SITE_DIR / "stack_download").is_dir():
    raise FileNotFoundError("open the notebook from the Guainia/ folder (stack_download/ not found)")
ROOT = next(p for p in [SITE_DIR, *SITE_DIR.parents] if (p / "biomass_tomo").is_dir())
sys.path.insert(0, str(ROOT))                       # the biomass_tomo package

import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="(All-NaN slice|Mean of empty slice|invalid value|divide by zero|Degrees of freedom)", category=RuntimeWarning)

from biomass_tomo import stack as st, tomo, reference as ref, display as dp

MANIFEST = SITE_DIR / "Guainia_outputs" / "run_manifest.json"
HEIGHT_CHANNEL = "X"           # channel used for the height maps: "X" (cross-pol), "HH" or "VV"
METHOD = "Capon"               # tomogram used by the profile estimators: "Capon" or "Beamformer"
COH_MIN = 0.1                  # quality mask: mean off-diagonal |gamma|
CONTRAST_MIN_DB = 1.0          # quality mask: profile max / mean [dB]
EDGE_LEVEL = 0.5               # min-max edge level
TOP_DB = -1.0                  # canopy-top threshold below the peak [dB]
H_MAX = 60.0                   # highest plausible canopy [m]
RUN_RVOG = True
RVOG_STEP = 2                  # RVoG on every RVOG_STEP-th cell in both directions (it is the slowest step)
WRITE_GEOTIFF = True

# reference forest height (None = no reference)
REFERENCE_HEIGHT = SITE_DIR / "Forest_Height_Guainia.tif"
REFERENCE_BAND = 1
REFERENCE_INVALID = (101, 102, 103)  # GLAD codes: water, snow/ice, no data
REFERENCE_NONFOREST = 0              # GLAD code: no trees
REFERENCE_POOL = (3, 3)              # pooling in coherence cells (≈ multilook footprint)
FOREST_MAX_NONFOREST = 0.2           # assessment mask: max fraction of tree-less 30 m pixels in the footprint
CORRECT_ESTIMATOR = None             # estimator to bias-correct; None = lowest RMSE against the reference
CORRECTION = "bias"                  # "bias" (subtract the mean bias) or "linear" (regress reference on estimate)
SPLIT_BLOCK_KM = 10.0                # spatial blocks (checkerboard) for the train / test split of the correction

cfg = st.read_manifest(MANIFEST)
if "tomography" not in cfg:
    raise RuntimeError("run notebook 2 first")
OUT = pathlib.Path(cfg["out_dir"]) / "forest_height"
OUT.mkdir(parents=True, exist_ok=True)
ref.fix_proj_data()

## 2. Data
Tomogram of `HEIGHT_CHANNEL`, $k_z$, coherence matrices and geolocation on the coherence grid. Maps are shown as (azimuth, range), with axes in ground kilometres.

In [ ]:
T = xr.open_dataset(cfg["tomography"]["tomograms"])
Z = T.height.values
vol = T[f"{METHOD.lower()}_{HEIGHT_CHANNEL}"].values
kz = T.kz.values
coh = np.load(cfg["tomography"]["coherence"][HEIGHT_CHANNEL])["coherence"]
REF = cfg["phase"]["reference_index"]
lat, lon, inc = T.latitude.values, T.longitude.values, T.incidence_angle.values
az_km = (T.azimuth.values - T.azimuth.values[0]) / 1e3
gr_km = dp.ground_range_km(T.range.values, inc)
mapview = lambda ax, v, **kw: dp.show(ax, np.asarray(v).T, az_km, gr_km, **kw)     # (range, azimuth) -> map
print(f"{METHOD} {HEIGHT_CHANNEL}: {vol.shape} (range, azimuth, height); window {T.attrs['window_m']} m; "
      f"resolution {cfg['tomography']['resolution_m']:.1f} m; scene {gr_km[-1]:.0f} km x {az_km[-1]:.0f} km")

## 3. Reference forest height on the radar grid
Every valid 30 m pixel is assigned to its nearest coherence cell (by the LUT geolocation). Sums are pooled over `REFERENCE_POOL` cells, which gives per cell:
- `ref_mean`: mean height of the 30 m pixels, with trees counted at their height and tree-less pixels as 0;
- `ref_max`: the highest 30 m pixel;
- `ref_std`: the spread of the 30 m pixels;
- the fractions of water/invalid and tree-less pixels.

The panels show the reference as delivered (lon/lat), on the radar grid, its within-footprint spread, and the forest mask used for the assessment.

In [ ]:
refmap, forest = {}, np.ones(lat.shape, bool)
if REFERENCE_HEIGHT is not None and pathlib.Path(REFERENCE_HEIGHT).exists():
    import rasterio
    t0 = time.time()
    refmap = ref.map_to_grid(REFERENCE_HEIGHT, lat, lon, band=REFERENCE_BAND, invalid=REFERENCE_INVALID,
                             nonforest=REFERENCE_NONFOREST, pool=REFERENCE_POOL)
    forest = (refmap["ref_invalid_frac"] == 0) & (refmap["ref_nonforest_frac"] <= FOREST_MAX_NONFOREST) & np.isfinite(refmap["ref_mean"])
    print(f"reference pooled in {time.time() - t0:.0f} s; median {np.nanmedian(refmap['ref_count']):.0f} pixels per footprint; "
          f"forest cells {forest.mean() * 100:.1f} %")
    print(f"ref_mean over forest: median {np.nanmedian(refmap['ref_mean'][forest]):.1f} m, "
          f"5-95 %: {np.nanpercentile(refmap['ref_mean'][forest], 5):.1f}-{np.nanpercentile(refmap['ref_mean'][forest], 95):.1f} m")

    with rasterio.open(REFERENCE_HEIGHT) as src:
        dec = 4
        native = src.read(REFERENCE_BAND, out_shape=(src.height // dec, src.width // dec)).astype(np.float32)
        b = src.bounds
    native[np.isin(native, REFERENCE_INVALID)] = np.nan
    hlo, hhi = dp.limits([native, refmap["ref_mean"][forest]], (1, 99))
    fig, axes = plt.subplots(1, 4, figsize=(17, 7.5), constrained_layout=True)
    im = axes[0].imshow(native, extent=[b.left, b.right, b.bottom, b.top], cmap="viridis", vmin=hlo, vmax=hhi, interpolation="none")
    axes[0].set_aspect(1 / np.cos(np.deg2rad(np.nanmean(lat)))); axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude")
    axes[0].set_title("reference (as delivered, lon/lat)", fontsize=9); fig.colorbar(im, ax=axes[0], shrink=0.6, label="m")
    im = mapview(axes[1], refmap["ref_mean"], cmap="viridis", vmin=hlo, vmax=hhi)
    axes[1].set_title("ref_mean on the radar grid", fontsize=9); axes[1].set_ylabel("azimuth [km]"); fig.colorbar(im, ax=axes[1], shrink=0.6, label="m")
    im = mapview(axes[2], refmap["ref_std"], cmap="magma", vmin=0, vmax=np.nanpercentile(refmap["ref_std"], 98))
    axes[2].set_title("ref_std within footprint", fontsize=9); fig.colorbar(im, ax=axes[2], shrink=0.6, label="m")
    mapview(axes[3], forest, cmap="gray", vmin=0, vmax=1); axes[3].set_title("forest mask (assessment)", fontsize=9)
    plt.show()
else:
    print("no reference height map: the assessment sections are skipped")

## 4. Quality mask
A cell has height information only if the stack is coherent (mean off-diagonal $|\gamma| \geq$ `COH_MIN`) and the profile has a peak (Capon max/mean ≥ `CONTRAST_MIN_DB`). Masked cells are typically water, frame edges (no-data) or strongly temporally decorrelated areas.

In [ ]:
nb = coh.shape[2]
mean_coh = np.nanmean(np.abs(coh)[:, :, ~np.eye(nb, dtype=bool)], axis=2)
capon_x = T[f"capon_{HEIGHT_CHANNEL}"].values
contrast = 10 * np.log10(np.nanmax(capon_x, axis=2) / np.nanmean(capon_x, axis=2))
reliable = tomo.quality_mask(coh, capon_x, COH_MIN, CONTRAST_MIN_DB)
print(f"reliable cells: {reliable.mean() * 100:.1f} %")
fig, axes = plt.subplots(1, 3, figsize=(13, 7.5), sharey=True, constrained_layout=True)
for ax, (v, title, kw) in zip(axes, [(mean_coh, "mean off-diagonal |γ|", dict(vmin=0, vmax=dp.limits(mean_coh)[1])),
                                     (contrast, "Capon contrast [dB]", dict(vmin=0, vmax=dp.limits(contrast)[1])),
                                     (reliable, "reliable", dict(vmin=0, vmax=1, cmap="gray"))]):
    im = mapview(ax, v, **{"cmap": "viridis", **kw})
    ax.set_title(title, fontsize=9); fig.colorbar(im, ax=ax, shrink=0.6)
axes[0].set_ylabel("azimuth [km]")
plt.show()

## 5. Height estimators

In [ ]:
t0 = time.time()
z_low, z_high = tomo.percentile_height(vol, Z, [5, 95], z_range=(Z[0], H_MAX + 10))
z_peak, z_edge = tomo.minmax_edge(vol, Z, level=EDGE_LEVEL, z_range=(Z[0], H_MAX + 10), z_top_max=H_MAX)
heights = {
    "H_envelope": z_high,
    "H_percentile": z_high - z_low,
    "H_minmax": z_edge - z_peak,
    "h_canopy_top": tomo.canopy_top(vol, Z, TOP_DB),
    "z_phase_centre": tomo.phase_centre(coh, kz, REF),
}
if RUN_RVOG:
    sl = (slice(None, None, RVOG_STEP), slice(None, None, RVOG_STEP))
    rv = tomo.rvog_inversion(coh[sl], kz[sl], inc[sl], valid=reliable[sl], hv_grid=np.arange(0, H_MAX + 1, 1.0))
    h_rv = np.full(reliable.shape, np.nan, np.float32)
    h_rv[sl] = np.where(rv["at_grid_limit"], np.nan, rv["h_v"])
    heights["h_rvog"] = h_rv
heights = {k: np.where(reliable & (v >= -5) & (v <= H_MAX), v, np.nan).astype(np.float32) for k, v in heights.items()}
print(f"computed in {time.time() - t0:.0f} s")
for k, v in heights.items():
    print(f"{k:16s} median {np.nanmedian(v):5.1f} m   5-95 %: {np.nanpercentile(v, 5):5.1f} - {np.nanpercentile(v, 95):5.1f} m   "
          f"valid {np.isfinite(v).mean() * 100:5.1f} %")

## 6. Maps and distributions
All estimators and the reference on one colour scale (2–98 % of all values). The histograms are over forest cells, so they compare directly with the reference.

In [ ]:
panels = dict(heights, **({"reference (ref_mean)": refmap["ref_mean"]} if refmap else {}))
vmin, vmax = dp.limits(list(panels.values()), (2, 98))
fig, axes = plt.subplots(1, len(panels), figsize=(2.6 * len(panels), 7.2), sharey=True, constrained_layout=True)
for ax, (k, v) in zip(axes, panels.items()):
    im = mapview(ax, v, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(k, fontsize=9)
axes[0].set_ylabel("azimuth [km]")
fig.colorbar(im, ax=axes, label="height [m]", shrink=0.6)
plt.show()

fig, ax = plt.subplots(figsize=(9, 3.6), constrained_layout=True)
bins = np.arange(-5, H_MAX + 1, 1)
for k, v in heights.items():
    ax.hist(v[forest & np.isfinite(v)], bins=bins, histtype="step", density=True, lw=1.4, label=k)
if refmap:
    ax.hist(refmap["ref_mean"][forest], bins=bins, color="k", alpha=0.25, density=True, label="reference ref_mean")
ax.set_xlabel("height [m]"); ax.set_ylabel("density"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

## 7. Example profiles
Random reliable forest cells: the Capon profile of `HEIGHT_CHANNEL` (dB, relative to its peak), with the estimated heights and the reference (`ref_mean` green, `ref_max` dotted green).

In [ ]:
rng = np.random.default_rng(0)
cand = np.argwhere(reliable & forest & np.isfinite(heights["h_canopy_top"]))
pick = cand[rng.choice(len(cand), size=min(8, len(cand)), replace=False)]
fig, axes = plt.subplots(1, len(pick), figsize=(2.1 * len(pick), 4.4), sharey=True, constrained_layout=True)
marks = {"h_canopy_top": ("r", "-"), "H_envelope": ("orange", "--"), "z_phase_centre": ("m", ":"), "h_rvog": ("c", "-.")}
for ax, (i, j) in zip(np.atleast_1d(axes), pick):
    p = vol[i, j]
    ax.plot(10 * np.log10(p / np.nanmax(p)), Z, "k")
    for k, (c, ls) in marks.items():
        if k in heights and np.isfinite(heights[k][i, j]):
            ax.axhline(heights[k][i, j], color=c, ls=ls, lw=1.2, label=k)
    if refmap:
        ax.axhline(refmap["ref_mean"][i, j], color="g", lw=2, alpha=0.7, label="reference mean")
        ax.axhline(refmap["ref_max"][i, j], color="g", ls=":", lw=1.5, label="reference max")
    ax.axhline(0, color="0.5", lw=0.6)
    ax.set_xlim(-15, 0.5); ax.set_title(f"({i}, {j})", fontsize=8); ax.set_xlabel("dB")
np.atleast_1d(axes)[0].set_ylabel("height [m]"); np.atleast_1d(axes)[-1].legend(fontsize=6, loc="upper right")
plt.show()

## 8. Assessment against the reference
Every estimator is compared with `ref_mean` over reliable forest cells: number of cells, correlation $r$, bias (estimate − reference), RMSE, MAE and the slope of estimate on reference. Note that over closed forest the reference spans only a few metres (5–95 %), so $r$ is limited by the reference itself. Bias and RMSE are the more informative numbers.

The figure shows
- **top row:** the joint distribution per estimator, with the 1:1 line (red dashed) and the binned median ± IQR (orange);
- **bottom row:** the residual against incidence angle. A trend here points to geometry ($k_z$, resolution) rather than forest.

In [ ]:
if refmap:
    R = refmap["ref_mean"]
    stats = {}
    for k, v in heights.items():
        s = ref.compare(R[forest], v[forest])
        if s:
            stats[k] = s
    table = pd.DataFrame({k: {m: s[m] for m in ("n", "r", "bias", "rmse", "mae", "slope")} for k, s in stats.items()}).T
    table = table.astype({"n": int}).sort_values("rmse")
    display(table.round(2))

    lo, hi = dp.limits(R[forest], (0.5, 99.5))
    rbins = np.arange(np.floor(lo), np.ceil(hi) + 1, 1.0)
    keys = list(table.index)
    fig, axes = plt.subplots(2, len(keys), figsize=(3.2 * len(keys), 6.6), constrained_layout=True)
    for col, k in enumerate(keys):
        ax = axes[0, col]
        x, y = R[forest], heights[k][forest]
        ok = np.isfinite(x) & np.isfinite(y)
        ax.hist2d(x[ok], y[ok], bins=[rbins, np.arange(-5, H_MAX + 1, 1)], cmap="Greys", cmin=1)
        cen = 0.5 * (rbins[1:] + rbins[:-1])
        q = np.array([np.nanpercentile(y[ok][(x[ok] >= a) & (x[ok] < b)], [25, 50, 75]) if ((x[ok] >= a) & (x[ok] < b)).sum() > 20
                      else [np.nan] * 3 for a, b in zip(rbins[:-1], rbins[1:])])
        ax.errorbar(cen, q[:, 1], yerr=[q[:, 1] - q[:, 0], q[:, 2] - q[:, 1]], fmt="o", color="tab:orange", ms=3, lw=1)
        ax.plot([0, H_MAX], [0, H_MAX], "r--", lw=0.8)
        ax.set_xlim(rbins[0], rbins[-1]); ax.set_ylim(-5, H_MAX)
        ax.set_xlabel("reference ref_mean [m]"); ax.set_ylabel(f"{k} [m]")
        s = stats[k]
        ax.set_title(f"r={s['r']:.2f} bias={s['bias']:+.1f} RMSE={s['rmse']:.1f}", fontsize=8)
        ax = axes[1, col]
        res = (heights[k] - R)[forest]
        ib = np.linspace(np.nanmin(inc), np.nanmax(inc), 16)
        cen_i = 0.5 * (ib[1:] + ib[:-1])
        q = np.array([np.nanpercentile(res[(inc[forest] >= a) & (inc[forest] < b)], [25, 50, 75]) for a, b in zip(ib[:-1], ib[1:])])
        ax.fill_between(cen_i, q[:, 0], q[:, 2], alpha=0.25); ax.plot(cen_i, q[:, 1], "o-", ms=3)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xlabel("incidence [deg]"); ax.set_ylabel("estimate − reference [m]"); ax.grid(alpha=0.3)
    plt.show()
else:
    table = None
    print("no reference: skipped")

### Difference map
The best estimator (lowest RMSE), the reference and their difference. A spatially coherent difference points to forest structure or terrain effects; salt-and-pepper differences are estimation noise.

In [ ]:
if refmap:
    BEST = CORRECT_ESTIMATOR or table.index[0]
    d = np.where(forest, heights[BEST] - refmap["ref_mean"], np.nan)
    fig, axes = plt.subplots(1, 3, figsize=(13, 7.5), sharey=True, constrained_layout=True)
    for ax, (v, title) in zip(axes[:2], ((heights[BEST], BEST), (refmap["ref_mean"], "reference ref_mean"))):
        im = mapview(ax, v, cmap="viridis", vmin=vmin, vmax=vmax); ax.set_title(title, fontsize=9)
    fig.colorbar(im, ax=axes[:2], shrink=0.6, label="height [m]")
    lim = np.nanpercentile(np.abs(d - np.nanmedian(d)), 95)
    im = mapview(axes[2], d, cmap="RdBu_r", vmin=np.nanmedian(d) - lim, vmax=np.nanmedian(d) + lim)
    axes[2].set_title(f"{BEST} − reference (forest)", fontsize=9); fig.colorbar(im, ax=axes[2], shrink=0.6, label="m")
    axes[0].set_ylabel("azimuth [km]")
    plt.show()

## 9. Bias correction
The best estimator still carries a systematic offset relative to the reference. With a 20 m vertical resolution, the −1 dB canopy-top threshold sits below the true top. The correction is fitted on half of the scene and tested on the other half. The halves are a checkerboard of `SPLIT_BLOCK_KM` blocks, so that neighbouring, correlated cells do not end up on both sides of the split.
- `CORRECTION = "bias"`: $h_{corr} = h - \overline{(h - h_{ref})}$. This keeps the spatial pattern of the radar estimate and is the default.
- `CORRECTION = "linear"`: $h_{corr} = a + b\,h$, the regression of the reference on the estimate. With a low $r$, this shrinks the map towards the reference mean ($b \ll 1$). It lowers the RMSE, but the variation then comes mostly from the reference.

In [ ]:
corrected = {}
if refmap:
    h, R = heights[BEST], refmap["ref_mean"]
    AZ, GR = np.meshgrid(az_km, gr_km)                                # (range, azimuth) like the maps
    train = ((np.floor(AZ / SPLIT_BLOCK_KM) + np.floor(GR / SPLIT_BLOCK_KM)) % 2 == 0)
    ok = forest & np.isfinite(h) & np.isfinite(R)
    tr, te = ok & train, ok & ~train
    if CORRECTION == "linear":
        b, a = np.polyfit(h[tr] - h[tr].mean(), R[tr], 1); a -= b * h[tr].mean()
        h_corr = a + b * h
        params = {"mode": "linear", "a": float(a), "b": float(b)}
    else:
        off = float(np.mean(h[tr] - R[tr]))
        h_corr = h - off
        params = {"mode": "bias", "offset_m": off}
    corrected = {f"{BEST}_corrected": h_corr.astype(np.float32)}
    rows = {"raw, test half": ref.compare(R[te], h[te]), "corrected, test half": ref.compare(R[te], h_corr[te])}
    display(pd.DataFrame({k: {m: s[m] for m in ("n", "r", "bias", "rmse", "mae")} for k, s in rows.items()}).T.round(2))
    print("correction", params, f"| train cells {tr.sum()}, test cells {te.sum()}")
    params.update(estimator=BEST, test_rmse_m=rows["corrected, test half"]["rmse"], test_bias_m=rows["corrected, test half"]["bias"])
else:
    params = None

## 10. Geographic maps and GeoTIFF
The height maps are placed on a regular lon/lat grid (EPSG:4326) by nearest neighbour from the LUT geolocation, at the coherence-grid spacing (≈ 95 m). Grid cells far from any radar cell are no-data. The GeoTIFF has one band per estimator, plus the corrected estimate and the pooled reference, with band descriptions.

In [ ]:
step = max(1, int(np.ceil(np.sqrt(lat.size / 400_000))))
warnings.filterwarnings("ignore", message="The input coordinates to pcolormesh", category=UserWarning)
geo = {f"{BEST if refmap else 'h_canopy_top'}": heights[BEST if refmap else "h_canopy_top"], **corrected,
       **({"reference ref_mean": refmap["ref_mean"]} if refmap else {})}
fig, axes = plt.subplots(1, len(geo), figsize=(5.2 * len(geo), 8), sharey=True, constrained_layout=True)
for ax, (k, v) in zip(np.atleast_1d(axes), geo.items()):
    pc = ax.pcolormesh(lon[::step, ::step], lat[::step, ::step], v[::step, ::step], cmap="viridis", vmin=vmin, vmax=vmax, shading="nearest")
    ax.set_aspect(1 / np.cos(np.deg2rad(np.nanmean(lat)))); ax.set_xlabel("longitude [deg]"); ax.set_title(k, fontsize=10)
np.atleast_1d(axes)[0].set_ylabel("latitude [deg]")
fig.colorbar(pc, ax=axes, label="height [m]", shrink=0.6)
plt.show()

if WRITE_GEOTIFF:
    tif = OUT / f"forest_height_{METHOD}_{HEIGHT_CHANNEL}_{cfg['tomography']['tag']}.tif"
    bands = {**heights, **corrected, **({"reference_ref_mean": refmap["ref_mean"]} if refmap else {})}
    _, res = ref.to_geotiff(bands, lat, lon, tif)
    print(f"GeoTIFF {tif} ({len(bands)} bands: {', '.join(bands)}; {res:.5f} deg ≈ {res * 111e3:.0f} m)")

## 11. Biomass (optional)
If the manifest names an AGBD map (`reference.agbd`, set in notebook 1), it is sampled at the cells and related to height.

In [ ]:
agbd = None
if cfg["reference"].get("agbd"):
    path, band = cfg["reference"]["agbd"]
    agbd = ref.sample_raster(path, lat, lon, band=band); agbd[agbd <= 0] = np.nan
    hb = heights[BEST if refmap else "h_canopy_top"]
    okm = np.isfinite(agbd) & np.isfinite(hb)
    if okm.sum() > 3:
        fig, ax = plt.subplots(figsize=(4.5, 4), constrained_layout=True)
        ax.hist2d(hb[okm], agbd[okm], bins=50, cmap="Greys", cmin=1)
        ax.set_xlabel("height [m]"); ax.set_ylabel("AGBD [Mg/ha]")
        ax.set_title(f"r = {np.corrcoef(hb[okm], agbd[okm])[0, 1]:.2f}, n = {okm.sum()}", fontsize=9)
        plt.show()
else:
    print("no AGBD map in the manifest: skipped")

## 12. Saving
Heights, the corrected estimate, the pooled reference layers, the quality and forest masks and the geolocation go to netCDF on the coherence grid. The manifest gains `forest_height`, with the assessment table and the correction.

In [ ]:
grid = ("range", "azimuth")
H = xr.Dataset({**{k: (grid, v) for k, v in heights.items()}, **{k: (grid, v) for k, v in corrected.items()},
                **{k: (grid, v) for k, v in refmap.items()},
                "forest": (grid, forest.astype(np.int8)),
                "z_peak": (grid, z_peak), "mean_coherence": (grid, mean_coh.astype(np.float32)),
                "capon_contrast_db": (grid, contrast.astype(np.float32)), "reliable": (grid, reliable.astype(np.int8)),
                **({"agbd": (grid, agbd)} if agbd is not None else {}),
                "latitude": (grid, lat), "longitude": (grid, lon), "incidence_angle": (grid, inc)},
               coords={"range": T.range.values, "azimuth": T.azimuth.values},
               attrs={**T.attrs, "height_channel": HEIGHT_CHANNEL, "method": METHOD, "edge_level": EDGE_LEVEL,
                      "top_db": TOP_DB, "coh_min": COH_MIN, "contrast_min_db": CONTRAST_MIN_DB, "rvog_step": RVOG_STEP if RUN_RVOG else 0,
                      "reference": str(REFERENCE_HEIGHT) if refmap else "", "reference_pool_cells": str(REFERENCE_POOL),
                      "units": "m above the SKP-calibrated ground"})
path = OUT / f"forest_height_{METHOD}_{HEIGHT_CHANNEL}_{cfg['tomography']['tag']}.nc"
st.save_netcdf(H, path)
cfg["reference"]["height"] = [str(REFERENCE_HEIGHT), REFERENCE_BAND] if refmap else None
cfg["forest_height"] = {"heights": str(path), "geotiff": str(tif) if WRITE_GEOTIFF else None,
                        "channel": HEIGHT_CHANNEL, "method": METHOD,
                        "median_m": {k: float(np.nanmedian(v)) for k, v in heights.items()},
                        "assessment": None if table is None else table.round(3).to_dict(orient="index"),
                        "correction": params}
st.write_manifest(MANIFEST, **cfg)
print("saved", path)